<a href="https://colab.research.google.com/github/bautistabc/AI_llama/blob/master/Copia_de_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.2 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [14]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto


In [15]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento: Mixto"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)

Few-shot: ¡Exacto! Los tres ejemplos que diste están correctamente clasificados:

| Reseña | Sentimiento |
|--------|-------------|
| *“Me encantó, llegó rápido y en perfecto estado.”* | Positivo |
| *“Nunca llegó mi pedido, pésimo servicio.”* | Negativo |
| *“El envío llegó tarde pero el producto es excelente.”* | Mixto |

Si necesitas clasificar más reseñas o revisar alguna otra, ¡solo dímelo!


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [16]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)

**Paso 1 – Comprender la situación**

- Tren A (el primero) sale de la ciudad **A** a una velocidad constante de **80 km/h**.  
- 2 horas después, Tren B (el segundo) sale también de la ciudad **A** pero a **120 km/h**.  
- Ambos se dirigen al mismo destino.

Queremos saber **cuánto tiempo (t) tarda el tren B en alcanzar al tren A**.

---

**Paso 2 – Identificar la brecha inicial**

Cuando el tren B inicia su viaje, el tren A ya ha avanzado 2 horas a 80 km/h:

\[
\text{Distancia inicial del tren A} = 80\;\text{km/h} \times 2\;\text{h} = 160\;\text{km}
\]

Así, el tren B está 160 km detrás cuando sale.

---

**Paso 3 – Determinar la velocidad relativa**

- Velocidad del tren A = 80 km/h.  
- Velocidad del tren B = 120 km/h.  

La velocidad con la que el tren B cierra la brecha es la diferencia:

\[
v_{\text{rel}} = 120\;\text{km/h} - 80\;\text{km/h} = 40\;\text{km/h}
\]

---

**Paso 4 – Calcular el tiempo necesario para cerrar la brecha**

El tiempo \(t\) que tarda el tren B en cubrir l

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [17]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Lo siento, no dispongo de esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [18]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [19]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [20]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [21]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)

No, los productos en oferta no son elegibles para devolución; solo se pueden cambiar de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [2]:
# Leer API key, instalar e importar librerías
!pip install -q sentence-transformers groq scikit-learn

from groq import Groq
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.8 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


In [3]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las concesiones de espectro radioeléctrico tienen una vigencia de 20 años y su renovación debe solicitarse al menos un año antes de su vencimiento.",
    "Toda homologación de equipo de telecomunicaciones requiere la presentación de un certificado de conformidad emitido por un laboratorio acreditado.",
    "Las solicitudes de interconexión entre operadores deben ser resueltas en un plazo no mayor a 60 días hábiles antes de la intervención del órgano regulador."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [5]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    #similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    similitudes = cosine_similarity(embedding_pregunta, embeddings_documentos)[0]
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo pedir una concesión con vigencia de 60 años?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Las concesiones de espectro radioeléctrico tienen una vigencia de 20 años y su renovación debe solicitarse al menos un año antes de su vencimiento.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [10]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
prompt_sin_rag = f"Responde a la siguiente pregunta: {pregunta}"

completion_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_sin_rag}],
    temperature=0.2,
)

respuesta_sin_rag = completion_sin_rag.choices[0].message.content

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [12]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
prompt_con_rag = f"""
Eres un asesor regulatorio de telecomunicaciones. Utiliza EXCLUSIVAMENTE el siguiente contexto para responder la pregunta. Si la respuesta no está en el contexto, di que no la sabes.

Contexto: {fragmento}

Pregunta: {pregunta}
"""

completion_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_con_rag}],
    temperature=0.0,
)

respuesta_con_rag = completion_con_rag.choices[0].message.content

**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [13]:
# Mostrar ambas respuestas para comparar
print("==================================================")
print("PREGUNTA:", pregunta)
print("==================================================\n")

print("--- 1. RESPUESTA SIN RAG ---")
print(respuesta_sin_rag)
print("\n" + "=" * 50 + "\n")

print("--- 2. RESPUESTA CON RAG ---")
print(respuesta_con_rag)
print("\n" + "=" * 50 + "\n")

PREGUNTA: ¿Puedo pedir una concesión con vigencia de 60 años?

--- 1. RESPUESTA SIN RAG ---
No.  
En la legislación española la vigencia máxima de una concesión es de **30 años**.  
Para la mayoría de los tipos de concesiones (servicios públicos, obras públicas, concesiones de uso de infraestructuras, etc.) la Ley 5/2012, de 27 de abril, de Concesiones de Servicios Públicos, establece que:

| Tipo de concesión | Duración máxima inicial | Posibilidad de prórroga |
|-------------------|------------------------|------------------------|
| Concesiones de servicios públicos | 30 años | Sí, hasta 30 años adicionales (máximo 60 años en total) |
| Concesiones de obras públicas | 30 años | Sí, hasta 30 años adicionales |
| Concesiones de uso de infraestructuras (e.g., puertos, aeropuertos) | 30 años | Sí, hasta 30 años adicionales |
| Concesiones de explotación minera | 30 años | Sí, hasta 30 años adicionales |

### ¿Qué significa esto en la práctica?

1. **No se puede solicitar una única conce